Add PDF
chunk it & embedd it
store it in a vector database
connect to the database
query the database using natural language through LLM

In [23]:
from dotenv import load_dotenv
from pathlib import Path
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

In [24]:
load_dotenv(Path.cwd() / ".env")

True

In [ ]:
file_path = "Test_doc.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()

In [71]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = text_splitter.split_documents(docs)

In [72]:
chunks[0].page_content

'Sai Krishna Reddy Poluri \nData Engineer'

In [73]:
len(chunks)

91

In [80]:
persist_directory = str(Path.cwd() / "chroma_db")
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory=persist_directory,
    collection_name="resume",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

In [86]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
question = "Who is Dalai Lama?"
docs = retriever.invoke(question)
context = "\n\n".join(d.page_content for d in docs)

from langchain_core.messages import SystemMessage, HumanMessage

llm_response = llm.invoke(
    [
        SystemMessage(
            content="Answer using only the context. If unknown, say you don't know.\n\nContext:\n"
            + context
        ),
        HumanMessage(content=question),
    ]
)
print(llm_response.content)

I don't know.


In [87]:
print(context)

Sai Krishna Reddy Poluri 
Data Engineer

PostgreSQL, SQL Server, MongoDB , CosmosDB (NoSQL)

Bachelor of Technology | SASTRA University, India 
CERTIFICATIONS

Data Engineer | University of Texas at Dallas | Richardson, TX

PROFESSIONAL SUMMARY

Architecture , maintaining full historical lineage  of

CERTIFICATIONS 
• Databricks Certified Data Engineer Associate

Data Analyst | Cognizant | India

Kafka, Hadoop, MapReduce, YARN, Hive, Sqoop

Data Analyst Intern | Redshift Technologies | Dallas, TX
